# Baselines: PromptShift-CRC and Domain-Shift-Aware CP (best-effort re-implementations)

**IMPORTANT HONESTY NOTE, read before trusting these numbers:**
Both methods below are very recent (2026) papers for which only the abstract /
high-level method description is publicly available to us -- not the authors'
code or exact hyperparameters. What follows is a **best-effort, clearly-documented
re-implementation of the core idea** each paper describes, adapted from text-only
LLMs to our multimodal setting. Treat results from this notebook as a first,
approximate comparison point, not a certified reproduction of either paper.
This caveat should be stated explicitly wherever these results are used.

- **Domain-Shift-Aware CP** (Lin et al., ICML 2026): reweights calibration
  examples by their embedding-space proximity to the current test item, then
  takes a weighted conformal quantile as the threshold.
- **PromptShift-CRC** (Opoku & Banahene, 2026): similar embedding-based
  reweighting, PLUS an online correction term that adapts the threshold after
  observed coverage violations (conceptually a hybrid of embedding-based
  reweighting and ACI-style online adaptation).

Neither paper is multimodal; we adapt both by embedding the **question text**
of each item (not the image) with a sentence-transformer, exactly the same
embedding model already used for our own hybrid score's semantic-volume term,
so the comparison is on equal footing.

**Before running:** Runtime -> Change runtime type -> GPU is NOT required for
this notebook (text embeddings only, no VLM calls) but doesn't hurt.
Also upload the five hybrid-score CSVs from your earlier runs:
`scores_mathvista_hybrid.csv`, `scores_ai2d_hybrid_fixed.csv`,
`scores_chartqa_hybrid.csv`, `scores_mmmu_hybrid.csv`, `scores_textvqa_hybrid.csv`
(Colab: use the Files panel on the left to upload them into /content/).

In [ ]:
!pip install -q sentence-transformers datasets

In [ ]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, concatenate_datasets

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedder loaded.")

DATASET_ORDER = [
    ("MathVista", "scores_mathvista_hybrid.csv"),
    ("AI2D",      "scores_ai2d_hybrid_fixed.csv"),
    ("ChartQA",   "scores_chartqa_hybrid.csv"),
    ("MMMU",      "scores_mmmu_hybrid.csv"),
    ("TextVQA",   "scores_textvqa_hybrid.csv"),
]

In [ ]:
# ------------------------------------------------------------------
# Reload the ORIGINAL question text for each item (cheap -- metadata
# only, no images needed, no model inference). Ids must match the
# "{dataset}_{i}" / "mmmu_validation_{subject}_{i}" scheme used when the
# scores were originally generated.
# ------------------------------------------------------------------

def load_questions_mathvista(n):
    ds = load_dataset("AI4Math/MathVista", split="testmini")
    out = {}
    for i, ex in enumerate(ds):
        if i >= n:
            break
        out[f"mathvista_{ex.get('pid', i)}"] = ex["question"]
    return out

def load_questions_ai2d(n):
    ds = load_dataset("lmms-lab/ai2d", split="test")
    out = {}
    for i, ex in enumerate(ds):
        if i >= n:
            break
        out[f"ai2d_{i}"] = ex["question"]
    return out

def load_questions_chartqa(n):
    ds = load_dataset("HuggingFaceM4/ChartQA", split="test")
    out = {}
    for i, ex in enumerate(ds):
        if i >= n:
            break
        out[f"chartqa_{i}"] = ex.get("query") or ex.get("question")
    return out

def load_questions_mmmu(n):
    subjects = ["Art", "Biology", "Computer_Science", "Math", "Physics", "Chemistry"]
    parts = []
    per_subj = max(1, n // len(subjects))
    for subj in subjects:
        try:
            d = load_dataset("MMMU/MMMU", subj, split="validation")
            parts.append(d.select(range(min(per_subj * 3, len(d)))))
        except Exception as e:
            print(f"[warn] skipping subject {subj}: {e}")
    ds = concatenate_datasets(parts) if parts else []
    out = {}
    count = 0
    for i, ex in enumerate(ds):
        if count >= n:
            break
        if ex.get("image_1") is None:
            continue
        out[f"mmmu_{ex.get('id', i)}"] = ex["question"]
        count += 1
    return out

def load_questions_textvqa(n):
    ds = load_dataset("lmms-lab-encoder/textvqa", split="validation")
    out = {}
    for i, ex in enumerate(ds):
        if i >= n:
            break
        out[f"textvqa_{i}"] = ex["question"]
    return out

QUESTION_LOADERS = {
    "MathVista": load_questions_mathvista,
    "AI2D": load_questions_ai2d,
    "ChartQA": load_questions_chartqa,
    "MMMU": load_questions_mmmu,
    "TextVQA": load_questions_textvqa,
}
print("Question loaders ready.")

In [ ]:
# ------------------------------------------------------------------
# Load the five hybrid-score CSVs, attach question text + embedding to
# each row, and build the combined 5-domain stream (same order as the
# paper's main result).
# ------------------------------------------------------------------

dfs = []
for name, fname in DATASET_ORDER:
    df = pd.read_csv(fname).sort_values("order").reset_index(drop=True)
    n_needed = len(df) * 3 if name == "MMMU" else len(df) + 5  # small buffer
    q_map = QUESTION_LOADERS[name](n_needed)
    df["question"] = df["id"].map(q_map)
    missing = df["question"].isna().sum()
    if missing:
        print(f"[warn] {name}: {missing}/{len(df)} items had no matching question text "
              f"(id-scheme mismatch) -- these will get a generic empty-string embedding.")
    df["question"] = df["question"].fillna("")
    df["domain"] = name
    dfs.append(df)
    print(f"{name}: {len(df)} rows, {missing} missing question text")

combined = pd.concat(dfs, ignore_index=True)
print(f"\nCombined stream: n={len(combined)}")

print("Embedding all questions (fast, CPU is fine) ...")
embeddings = embedder.encode(combined["question"].tolist(), normalize_embeddings=True, show_progress_bar=True)
print("Done. Embedding shape:", embeddings.shape)

In [ ]:
# ------------------------------------------------------------------
# Baseline methods (best-effort re-implementations -- see the honesty
# note at the top of this notebook).
# ------------------------------------------------------------------

ALPHA = 0.30

def weighted_quantile(values, weights, q):
    # Weighted quantile via sorted cumulative-weight interpolation.
    order = np.argsort(values)
    v = values[order]
    w = weights[order]
    cw = np.cumsum(w)
    cw /= cw[-1]
    idx = np.searchsorted(cw, q)
    idx = min(idx, len(v) - 1)
    return v[idx]


def domain_shift_aware_cp(scores, embeddings, alpha=ALPHA, calib_n=30, temperature=0.5,
                           max_calib_pool=300):
    # Best-effort re-implementation of the core idea in Lin et al. (2026),
    # "Domain-Shift-Aware Conformal Prediction for Large Language Models":
    # at each test step t, weight the available calibration pool (the initial
    # calibration window, expanded online with all previously-seen items, up
    # to max_calib_pool most recent to bound compute) by the cosine similarity
    # between the current item's embedding and each calibration item's
    # embedding (softmax-normalized with a temperature), and take the
    # (1-alpha) WEIGHTED quantile of the calibration scores as the threshold.
    # Unlike our method (Sec. 3.3), there is no separate scalar step size --
    # all adaptation happens through the similarity-based reweighting itself.
    T = len(scores)
    err = np.zeros(T)
    q = np.zeros(T)
    pool_idx = list(range(calib_n))
    q[0] = np.quantile(scores[:calib_n], 1 - alpha)
    for t in range(T):
        pool = pool_idx[-max_calib_pool:]
        sims = embeddings[pool] @ embeddings[t]
        w = np.exp(sims / temperature)
        w /= w.sum()
        q_t = weighted_quantile(scores[pool], w, 1 - alpha)
        q[t] = q_t
        err[t] = float(scores[t] > q_t)
        pool_idx.append(t if t >= calib_n else None)
        pool_idx = [p for p in pool_idx if p is not None]
    return err, q


def promptshift_crc(scores, embeddings, alpha=ALPHA, calib_n=30, temperature=0.5,
                     max_calib_pool=300, eta=0.02):
    # Best-effort re-implementation of the core idea in Opoku & Banahene
    # (2026), "PromptShift-CRC": like domain_shift_aware_cp above (embedding-
    # similarity-weighted calibration quantile), PLUS an online additive
    # correction term updated after each observed violation (their "adapts
    # the risk threshold online after observed violations" description),
    # combining reweighting with ACI-style online tracking.
    T = len(scores)
    err = np.zeros(T)
    q = np.zeros(T)
    pool_idx = list(range(calib_n))
    correction = 0.0
    q0 = np.quantile(scores[:calib_n], 1 - alpha)
    q[0] = q0
    for t in range(T):
        pool = pool_idx[-max_calib_pool:]
        sims = embeddings[pool] @ embeddings[t]
        w = np.exp(sims / temperature)
        w /= w.sum()
        base_q = weighted_quantile(scores[pool], w, 1 - alpha)
        q_t = max(0.0, base_q + correction)
        q[t] = q_t
        err[t] = float(scores[t] > q_t)
        correction += eta * (err[t] - alpha)
        pool_idx.append(t if t >= calib_n else None)
        pool_idx = [p for p in pool_idx if p is not None]
    return err, q


print("Baseline methods defined.")

In [ ]:
# ------------------------------------------------------------------
# Run both new baselines on the combined stream, alongside our own
# four methods (re-defined inline here to keep this notebook
# self-contained; identical to calibration/calibration_methods.py in
# the code repository).
# ------------------------------------------------------------------

def fixed_cp(scores, alpha=ALPHA, calib_n=30):
    q0 = np.quantile(scores[:calib_n], 1 - alpha)
    return (scores > q0).astype(float), np.full(len(scores), q0)

def standard_aci(scores, alpha=ALPHA, calib_n=30, gamma=0.03):
    T = len(scores); q0 = np.quantile(scores[:calib_n], 1 - alpha)
    q = np.zeros(T); q[0] = q0; err = np.zeros(T)
    for t in range(T):
        err[t] = float(scores[t] > q[t])
        if t + 1 < T:
            q[t+1] = max(0.0, q[t] + gamma * (err[t] - alpha))
    return err, q

def drift_aware_aci(scores, alpha=ALPHA, calib_n=30, gamma0=0.03, lam=4.0,
                     gamma_min=0.005, gamma_max=0.30, window=25):
    T = len(scores); calib = scores[:calib_n]; q0 = np.quantile(calib, 1 - alpha)
    cm, cs = calib.mean(), calib.std() + 1e-6
    q = np.zeros(T); q[0] = q0; err = np.zeros(T)
    for t in range(T):
        err[t] = float(scores[t] > q[t])
        lo = max(0, t - window); recent = scores[lo:t+1].mean()
        D_t = abs(recent - cm) / cs
        g = np.clip(gamma0 * (1 + lam * D_t), gamma_min, gamma_max)
        if t + 1 < T:
            q[t+1] = max(0.0, q[t] + g * (err[t] - alpha))
    return err, q

scores = combined["nonconformity_score"].values.astype(float)
T = len(scores)
boundaries = np.cumsum([len(d) for d in dfs])
switch_points = boundaries[:-1]
domain_names = [n for n, _ in DATASET_ORDER]
domain_bounds = list(zip([0] + list(switch_points), list(switch_points) + [T]))

def coverage(err, a=None, b=None):
    seg = err[a:b] if (a is not None or b is not None) else err
    return 1 - seg.mean()

METHODS_TO_RUN = {
    "Fixed / Split CP": lambda: fixed_cp(scores),
    "Standard ACI": lambda: standard_aci(scores),
    "Drift-Aware ACI (ours)": lambda: drift_aware_aci(scores),
    "Domain-Shift-Aware CP [9] (re-impl.)": lambda: domain_shift_aware_cp(scores, embeddings),
    "PromptShift-CRC [8] (re-impl.)": lambda: promptshift_crc(scores, embeddings),
}

print(f"target coverage = {1-ALPHA:.2f}\n")
results = {}
for name, fn in METHODS_TO_RUN.items():
    err, q = fn()
    row = [coverage(err)] + [coverage(err, a, b) for a, b in domain_bounds]
    results[name] = row
    print(f"{name:<38}" + "  ".join(f"{v:.3f}" for v in row))

rdf = pd.DataFrame(results, index=["Overall"] + domain_names).T
rdf.to_csv("baseline_comparison_results.csv")
print("\nSaved: baseline_comparison_results.csv")
rdf

## Download the results

Download `baseline_comparison_results.csv` (Colab: Files panel on the left,
or `from google.colab import files; files.download(...)`) and send it back
for the final paper comparison table and honest write-up of these results.

In [ ]:
# from google.colab import files
# files.download("baseline_comparison_results.csv")